# Rezervácia hotela s middleware pre prioritných členov

Tento notebook demonštruje **middleware založené na funkciách** pomocou Microsoft Agent Framework. Staviame na príklade podmieneného workflow pridaním middleware vrstvy, ktorá dáva prioritným členom špeciálne výhody.

## Čo sa naučíte:
1. **Middleware založené na funkciách**: Zachytávať a upravovať výsledky funkcií
2. **Prístup ku kontextu**: Čítať a meniť `context.result` po vykonaní
3. **Implementácia obchodnej logiky**: Výhody pre prioritných členov
4. **Zmena výsledku**: Zmeniť výstup funkcie podľa statusu používateľa
5. **Rovnaký workflow, rôzne výsledky**: Zmeny správania riadené middleware

## Architektúra workflow s middleware:

```
User Input: "I want to book a hotel in Paris"
                    ↓
        [availability_agent]
        - Calls hotel_booking tool
        - 🌟 priority_check middleware intercepts
        - Checks user membership status
        - IF priority + no rooms → Override to available!
        - Returns BookingCheckResult
                    ↓
        Conditional Routing
           /                    \
    [has_availability]    [no_availability]
          ↓                      ↓
    [booking_agent]        [alternative_agent]
    (Priority override!)   (Regular users)
          ↓                      ↓
       [display_result executor]
```

## Kľúčový rozdiel oproti podmienenému workflow:

**Bez middleware** (14-conditional-workflow.ipynb):
- Paríž nemá voľné izby → presmerovať na alternative_agent

**S middleware** (tento notebook):
- Bežný používateľ + Paríž → žiadne voľné izby → presmerovať na alternative_agent
- Prioritný používateľ + Paríž → 🌟 Middleware prepíše! → dostupné → presmerovať na booking_agent

## Predpoklady:
- Nainštalovaný Microsoft Agent Framework
- Pochopenie podmienených workflow (pozri 14-conditional-workflow.ipynb)
- Token GitHub alebo OpenAI API kľúč
- Základné pochopenie vzorov middleware


In [ ]:
import asyncio
import json
import os
from collections.abc import Awaitable, Callable
from typing import Annotated, Any, Never

from agent_framework import (
    AgentExecutor,
    AgentExecutorRequest,
    AgentExecutorResponse,
    FunctionInvocationContext,
    Message,
    WorkflowBuilder,
    WorkflowContext,
    executor,
    tool,
)
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential
from dotenv import load_dotenv
from IPython.display import HTML, display
from pydantic import BaseModel

print("✅ All imports successful!")

## Krok 1: Definujte Pydantic modely pre štruktúrované výstupy

Tieto modely definujú **schému**, ktorú agenti vrátia. Pridali sme pole `priority_override`, aby sme sledovali, kedy middleware upravuje výsledok dostupnosti.


In [ ]:
class BookingCheckResult(BaseModel):
    """Result from checking hotel availability at a destination."""

    destination: str
    has_availability: bool
    message: str
    # Tracks if middleware overrode the result. The Azure structured-output
    # contract requires every property to be in the JSON schema's `required`
    # array, so we cannot give this a default value the way the original
    # notebook did.
    priority_override: bool


class AlternativeResult(BaseModel):
    """Suggested alternative destination when no rooms available."""

    alternative_destination: str
    reason: str


class BookingConfirmation(BaseModel):
    """Booking suggestion when rooms are available."""

    destination: str
    action: str
    message: str


print("✅ Pydantic models defined:")
print("   - BookingCheckResult (availability check with priority_override)")
print("   - AlternativeResult (alternative suggestion)")
print("   - BookingConfirmation (booking confirmation)")

## Krok 2: Definujte databázu prioritných členov

Pre túto ukážku budeme simulovať databázu prioritných členov. V produkcii by sa dotazovalo na skutočnú databázu alebo API.

**Prioritní členovia:**
- `alice@example.com` - člen VIP
- `bob@example.com` - prémiový člen  
- `priority_user` - testovací účet


In [ ]:
# Simulated priority members database
PRIORITY_MEMBERS = {
    "alice@example.com",
    "bob@example.com",
    "priority_user",
}

# Global variable to track current user (in real app, use proper session management)
current_user_id = "regular_user"  # Default: regular user


def set_user(user_id: str):
    """Set the current user for the session."""
    global current_user_id
    current_user_id = user_id
    is_priority = user_id in PRIORITY_MEMBERS
    status = "🌟 PRIORITY MEMBER" if is_priority else "👤 Regular User"

    display(
        HTML(f"""
        <div style='padding: 15px; background: {"linear-gradient(135deg, #FFD700 0%, #FFA500 100%)" if is_priority else "#e3f2fd"}; 
                    border-left: 4px solid {"#FF6B35" if is_priority else "#2196f3"}; border-radius: 4px; margin: 10px 0;'>
            <strong>👤 Current User Set:</strong> {user_id}<br>
            <strong>Status:</strong> {status}
        </div>
    """)
    )


print("✅ Priority members database created")
print(f"   Priority members: {len(PRIORITY_MEMBERS)} users")

## Krok 3: Vytvorte nástroj na rezerváciu hotela

Rovnako ako podmienený pracovný proces, ale teraz ho bude zachytávať middleware!


In [ ]:
@tool(description="Check hotel room availability for a destination city")
def hotel_booking(destination: Annotated[str, "The destination city to check for hotel rooms"]) -> str:
    """
    Simulates checking hotel room availability.

    Returns JSON string with availability status.
    """
    display(
        HTML(f"""
        <div style='padding: 15px; background: #e3f2fd; border-left: 4px solid #2196f3; border-radius: 4px; margin: 10px 0;'>
            <strong>🔍 Tool Invoked:</strong> hotel_booking("{destination}")
        </div>
    """)
    )

    # Simulate availability check
    cities_with_rooms = ["stockholm", "seattle", "tokyo", "london", "amsterdam"]
    has_rooms = destination.lower() in cities_with_rooms

    result = {"has_availability": has_rooms, "destination": destination}

    return json.dumps(result)


print("✅ hotel_booking tool created with @tool decorator")

## Krok 4: 🌟 Vytvorte Priority Check Middleware (HLAVNÁ FUNKCIA!)

Toto je **jadrová funkcionalita** tohto poznámkového bloku. Middleware:

1. **Zachytáva** volanie funkcie hotel_booking
2. **Vykonáva** funkciu bežne volaním `next(context)`
3. **Skúma** výsledok v `context.result`
4. **Prepíše** výsledok, ak je používateľ prioritný a nie sú dostupné žiadne izby
5. **Vráti** upravený výsledok späť agentovi

**Kľúčový vzor:**
```python
async def my_middleware(context, next):
    await next(context)  # Vykonať funkciu
    # Teraz context.result obsahuje výstup funkcie
    if some_condition:
        context.result = new_value  # Prepísať!
```


In [ ]:
async def priority_check_middleware(
    context: FunctionInvocationContext,
    next: Callable[[FunctionInvocationContext], Awaitable[None]],
) -> None:
    """
    Function middleware that overrides hotel_booking results for priority members.
    
    Workflow:
    1. Let the function execute normally
    2. Check if user is a priority member
    3. If priority + no availability → Override to make rooms available!
    4. Agent will then route to booking path instead of alternative path
    """
    function_name = context.function.name

    display(
        HTML(f"""
        <div style='padding: 12px; background: #fff3e0; border-left: 4px solid #ff9800; border-radius: 4px; margin: 10px 0;'>
            <strong>🔄 Middleware:</strong> Intercepting {function_name}...
        </div>
    """)
    )

    # Execute the original function
    await next(context)

    # Now inspect and potentially modify the result
    if context.result and function_name == "hotel_booking":
        result_data = json.loads(context.result)
        destination = result_data.get("destination", "")
        has_availability = result_data.get("has_availability", False)

        # Check if user is priority member
        is_priority = current_user_id in PRIORITY_MEMBERS

        # Override logic: Priority member + no availability → Make available!
        if is_priority and not has_availability:
            display(
                HTML(f"""
                <div style='padding: 20px; background: linear-gradient(135deg, #FFD700 0%, #FFA500 100%); 
                            border-radius: 8px; margin: 10px 0; box-shadow: 0 4px 12px rgba(255,165,0,0.4);'>
                    <h3 style='margin: 0 0 10px 0; color: #333;'>🌟 PRIORITY OVERRIDE ACTIVATED! 🌟</h3>
                    <p style='margin: 0; color: #555; line-height: 1.6;'>
                        <strong>User:</strong> {current_user_id}<br>
                        <strong>Status:</strong> VIP Priority Member<br>
                        <strong>Action:</strong> Overriding "No Availability" for {destination}<br>
                        <strong>Result:</strong> ✅ Rooms now available for priority booking!
                    </p>
                </div>
            """)
            )

            # Override the result!
            result_data["has_availability"] = True
            result_data["priority_override"] = True
            context.result = json.dumps(result_data)

        elif not has_availability:
            display(
                HTML(f"""
                <div style='padding: 12px; background: #ffebee; border-left: 4px solid #f44336; border-radius: 4px; margin: 10px 0;'>
                    <strong>ℹ️ Middleware:</strong> No priority override (user: {current_user_id})
                </div>
            """)
            )


print("✅ priority_check_middleware created")
print("   - Intercepts hotel_booking function")
print("   - Overrides availability for priority members")

## Krok 5: Definujte funkcie podmienok pre smerovanie

Rovnako ako funkcie podmienok v podmienenom pracovnom postupe – kontrolujú štruktúrovaný výstup na určenie smerovania.


In [ ]:
def has_availability_condition(message: Any) -> bool:
    """Condition for routing when hotels ARE available (including priority overrides!)."""
    if not isinstance(message, AgentExecutorResponse):
        return True

    try:
        result = BookingCheckResult.model_validate_json(message.agent_run_response.text)

        # Check if this was a priority override
        override_indicator = " 🌟" if result.priority_override else ""

        display(
            HTML(f"""
            <div style='padding: 12px; background: #c8e6c9; border-left: 4px solid #4caf50; border-radius: 4px; margin: 10px 0;'>
                <strong>✅ Condition Check:</strong> has_availability = <strong>{result.has_availability}</strong> for {result.destination}{override_indicator}
            </div>
        """)
        )

        return result.has_availability
    except Exception as e:
        display(
            HTML(f"""
            <div style='padding: 12px; background: #ffcdd2; border-left: 4px solid #f44336; border-radius: 4px; margin: 10px 0;'>
                <strong>⚠️  Error:</strong> {str(e)}
            </div>
        """)
        )
        return False


def no_availability_condition(message: Any) -> bool:
    """Condition for routing when hotels are NOT available."""
    if not isinstance(message, AgentExecutorResponse):
        return False

    try:
        result = BookingCheckResult.model_validate_json(message.agent_run_response.text)

        display(
            HTML(f"""
            <div style='padding: 12px; background: #ffecb3; border-left: 4px solid #ff9800; border-radius: 4px; margin: 10px 0;'>
                <strong>❌ Condition Check:</strong> no_availability for {result.destination}
            </div>
        """)
        )

        return not result.has_availability
    except Exception:
        return False


print("✅ Condition functions defined")

## Krok 6: Vytvorenie vlastného vykonávateľa zobrazenia

Rovnaký vykonávateľ ako predtým - zobrazuje konečný výstup pracovného postupu.


In [ ]:
@executor(id="display_result")
async def display_result(response: AgentExecutorResponse, ctx: WorkflowContext[Never, str]) -> None:
    """Display the final result as workflow output."""
    display(
        HTML("""
        <div style='padding: 15px; background: #f3e5f5; border-left: 4px solid #9c27b0; border-radius: 4px; margin: 10px 0;'>
            <strong>📤 Display Executor:</strong> Yielding workflow output
        </div>
    """)
    )

    await ctx.yield_output(response.agent_run_response.text)


print("✅ display_result executor created")

## Krok 7: Načítanie premenných prostredia

Nakonfigurujte LLM klienta (GitHub Models alebo OpenAI).


In [ ]:
# Load environment variables
load_dotenv()

# Configure the Azure AI Foundry provider with keyless authentication
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Krok 8: Vytvorenie AI agentov s middleware

**HLAVNÝ ROZDIEL:** Pri vytváraní availability_agent odovzdávame parameter `middleware`!

Takto vkladáme priority_check_middleware do volacieho potrubia funkcie agenta.


In [ ]:
# Agent 1: Check availability with tool + middleware
availability_agent = AgentExecutor(
    await provider.create_agent(
        name="availability-agent",
        instructions=(
            "You are a hotel booking assistant that checks room availability. "
            "Use the hotel_booking tool to check if rooms are available at the destination. "
            "Return JSON with fields: destination (string), has_availability (bool), message (string), "
            "and priority_override (bool, true if priority member got special access). "
            "The message should summarize the availability status and mention if priority override occurred."
        ),
        tools=[hotel_booking],
        default_options={"response_format": BookingCheckResult},
        middleware=[priority_check_middleware],  # 🌟 MIDDLEWARE INJECTION!
    ),
    id="availability_agent",
)

# Agent 2: Suggest alternative (when no rooms)
alternative_agent = AgentExecutor(
    await provider.create_agent(
        name="alternative-agent",
        instructions=(
            "You are a helpful travel assistant. When a user cannot find hotels in their requested city, "
            "suggest an alternative nearby city that has availability. "
            "Return JSON with fields: alternative_destination (string) and reason (string). "
            "Make your suggestion sound appealing and helpful."
        ),
        default_options={"response_format": AlternativeResult},
    ),
    id="alternative_agent",
)

# Agent 3: Suggest booking (when rooms available)
booking_agent = AgentExecutor(
    await provider.create_agent(
        name="booking-agent",
        instructions=(
            "You are a booking assistant. The user has found available hotel rooms. "
            "Encourage them to book by highlighting the destination's appeal. "
            "If priority_override is true in the input, mention that they received priority member access. "
            "Return JSON with fields: destination (string), action (string), and message (string). "
            "The action should be 'book_now' and message should be encouraging."
        ),
        default_options={"response_format": BookingConfirmation},
    ),
    id="booking_agent",
)

display(
    HTML("""
    <div style='padding: 15px; background: #e3f2fd; border-left: 4px solid #2196f3; border-radius: 4px; margin: 10px 0;'>
        <strong>✅ Created 3 Agents:</strong>
        <ul style='margin: 10px 0 0 0;'>
            <li><strong>availability_agent</strong> - WITH priority_check_middleware 🌟</li>
            <li><strong>alternative_agent</strong> - Suggests alternative cities</li>
            <li><strong>booking_agent</strong> - Encourages booking</li>
        </ul>
    </div>
""")
)

## Krok 9: Vytvorenie pracovného postupu

Rovnaká štruktúra pracovného postupu ako predtým - podmienené smerovanie na základe dostupnosti.


In [ ]:
# Build the workflow with conditional routing
workflow = (
    WorkflowBuilder(
        start_executor=availability_agent,
        output_executors=[display_result],
    )
    # NO AVAILABILITY PATH
    .add_edge(availability_agent, alternative_agent, condition=no_availability_condition)
    .add_edge(alternative_agent, display_result)
    # HAS AVAILABILITY PATH (can be triggered by middleware override!)
    .add_edge(availability_agent, booking_agent, condition=has_availability_condition)
    .add_edge(booking_agent, display_result)
    .build()
)

display(
    HTML("""
    <div style='padding: 20px; background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); color: white; border-radius: 8px; margin: 10px 0;'>
        <h3 style='margin: 0 0 15px 0;'>✅ Workflow Built Successfully!</h3>
        <p style='margin: 0; line-height: 1.6;'>
            <strong>Conditional Routing (Middleware-Aware):</strong><br>
            • If <strong>NO availability</strong> → alternative_agent → display_result<br>
            • If <strong>availability</strong> (or 🌟 <strong>priority override</strong>) → booking_agent → display_result
        </p>
    </div>
""")
)

## Krok 10: Testovací prípad 1 - Bežný používateľ v Paríži (Bez prepísania)

Bežný používateľ sa snaží zarezervovať Paríž → Žiadne izby → Presmeruje na alternative_agent


In [ ]:
# Set as regular user
set_user("regular_user")

display(
    HTML("""
    <div style='padding: 20px; background: #fff3e0; border-left: 4px solid #ff9800; border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #e65100;'>🧪 TEST CASE 1: Regular User + Paris</h3>
        <p style='margin: 0;'><strong>Expected:</strong> No rooms → No middleware override → Alternative suggestion</p>
    </div>
""")
)

# Create request
request_regular = AgentExecutorRequest(
    messages=[Message(role="user", text="I want to book a hotel in Paris")], should_respond=True
)

# Run workflow
events_regular = await workflow.run(request_regular)
outputs_regular = events_regular.get_outputs()

# Display results
if outputs_regular:
    result_regular = AlternativeResult.model_validate_json(outputs_regular[0])

    display(
        HTML(f"""
        <div style='padding: 25px; background: #fff; border: 2px solid #ff9800; border-radius: 12px; margin: 20px 0;'>
            <h3 style='margin: 0 0 15px 0; color: #e65100;'>📊 RESULT (Regular User)</h3>
            <div style='background: #fff3e0; padding: 20px; border-radius: 8px;'>
                <p style='margin: 0 0 10px 0;'><strong>Status:</strong> ❌ No rooms in Paris</p>
                <p style='margin: 0 0 10px 0;'><strong>Middleware:</strong> No priority override (regular user)</p>
                <p style='margin: 0 0 10px 0;'><strong>Alternative:</strong> 🏨 {result_regular.alternative_destination}</p>
                <p style='margin: 0;'><strong>Reason:</strong> {result_regular.reason}</p>
            </div>
        </div>
    """)
    )

## Krok 11: Testovací prípad 2 - 🌟 Prioritný používateľ v Paríži (S PREPÍSANÍM!)

Prioritný člen sa snaží zarezervovať Paríž → Spočiatku žiadne izby → 🌟 Middleware prepisuje! → Smeruje na booking_agent

**Toto je kľúčová ukážka sily middleware!**


In [ ]:
# Set as priority user
set_user("priority_user")

display(
    HTML("""
    <div style='padding: 20px; background: linear-gradient(135deg, #FFD700 0%, #FFA500 100%); border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #333;'>🧪 TEST CASE 2: 🌟 Priority User + Paris</h3>
        <p style='margin: 0; color: #555;'><strong>Expected:</strong> No rooms → 🌟 MIDDLEWARE OVERRIDE → Rooms available → Booking suggestion!</p>
    </div>
""")
)

# Create request
request_priority = AgentExecutorRequest(
    messages=[Message(role="user", text="I want to book a hotel in Paris")], should_respond=True
)

# Run workflow
events_priority = await workflow.run(request_priority)
outputs_priority = events_priority.get_outputs()

# Display results
if outputs_priority:
    result_priority = BookingConfirmation.model_validate_json(outputs_priority[0])

    display(
        HTML(f"""
        <div style='padding: 25px; background: linear-gradient(135deg, #FFD700 0%, #FFA500 100%); border-radius: 12px;
                    box-shadow: 0 8px 16px rgba(255,165,0,0.4); margin: 20px 0;'>
            <h3 style='margin: 0 0 15px 0; color: #333;'>🏆 RESULT (Priority Member) 🌟</h3>
            <div style='background: white; padding: 20px; border-radius: 8px;'>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ✅ Rooms Available (Priority Override!)</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Middleware:</strong> 🌟 OVERRIDE ACTIVATED!</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Destination:</strong> 🏨 {result_priority.destination}</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Action:</strong> {result_priority.action}</p>
                <p style='margin: 0; font-size: 14px; color: #666;'><strong>Message:</strong> {result_priority.message}</p>
                <div style='margin-top: 15px; padding: 15px; background: #fff3cd; border-radius: 6px; border-left: 4px solid #FF6B35;'>
                    <strong>💡 What Just Happened:</strong><br>
                    1. hotel_booking tool returned "no availability"<br>
                    2. priority_check_middleware intercepted the result<br>
                    3. Middleware checked user status: priority_user ✅<br>
                    4. Middleware OVERRODE the result to "available"<br>
                    5. Workflow routed to booking_agent instead of alternative_agent!
                </div>
            </div>
        </div>
    """)
    )

## Krok 12: Testovací prípad 3 - Prioritný používateľ v Štokholme (už dostupný)

Prioritný používateľ skúša Štokholm → Izby dostupné → Nie je potrebné prepísať → Presmeruje na booking_agent

Toto ukazuje, že middleware zasahuje len v prípade potreby!


In [ ]:
# Priority user is still set from previous test

display(
    HTML("""
    <div style='padding: 20px; background: #e8f5e9; border-left: 4px solid #4caf50; border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #1b5e20;'>🧪 TEST CASE 3: Priority User + Stockholm</h3>
        <p style='margin: 0;'><strong>Expected:</strong> Rooms available → No override needed → Booking suggestion</p>
    </div>
""")
)

# Create request
request_stockholm = AgentExecutorRequest(
    messages=[Message(role="user", text="I want to book a hotel in Stockholm")], should_respond=True
)

# Run workflow
events_stockholm = await workflow.run(request_stockholm)
outputs_stockholm = events_stockholm.get_outputs()

# Display results
if outputs_stockholm:
    result_stockholm = BookingConfirmation.model_validate_json(outputs_stockholm[0])

    display(
        HTML(f"""
        <div style='padding: 25px; background: linear-gradient(135deg, #4caf50 0%, #8bc34a 100%); color: white; border-radius: 12px;
                    box-shadow: 0 4px 12px rgba(76,175,80,0.3); margin: 20px 0;'>
            <h3 style='margin: 0 0 15px 0;'>🏆 RESULT (Priority User - No Override Needed)</h3>
            <div style='background: white; color: #333; padding: 20px; border-radius: 8px;'>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ✅ Rooms Available (Natural)</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Middleware:</strong> No override needed</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Destination:</strong> 🏨 {result_stockholm.destination}</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Action:</strong> {result_stockholm.action}</p>
                <p style='margin: 0; font-size: 14px; color: #666;'><strong>Message:</strong> {result_stockholm.message}</p>
                <div style='margin-top: 15px; padding: 15px; background: #e8f5e9; border-radius: 6px; border-left: 4px solid #4caf50;'>
                    <strong>💡 Middleware Behavior:</strong><br>
                    • hotel_booking returned "available" naturally<br>
                    • Middleware saw available = true → No override needed<br>
                    • Workflow proceeded normally to booking_agent
                </div>
            </div>
        </div>
    """)
    )

## Kľúčové poznatky a koncepty middleware

### ✅ Čo ste sa naučili:

#### **1. Middleware založený na funkciách**

Middleware zachytáva volania funkcií pomocou jednoduchej asynchrónnej funkcie:

```python
async def my_middleware(
    context: FunctionInvocationContext,
    next: Callable,
) -> None:
    # Pred vykonaním funkcie
    print("Intercepting...")
    
    # Vykonajte funkciu
    await next(context)
    
    # Po vykonaní funkcie - skontrolujte výsledok
    if context.result:
        # V prípade potreby upravte výsledok
        context.result = modified_value
```

#### **2. Prístup ku kontextu a prepis výsledku**

- `context.function` - Prístup k volanej funkcii
- `context.arguments` - Načítanie argumentov funkcie
- `context.kwargs` - Prístup k ďalším parametrom
- `await next(context)` - Vykonanie funkcie
- `context.result` - Čítanie/úprava výstupu funkcie

#### **3. Implementácia biznis logiky**

Náš middleware implementuje výhody pre uprednostnených členov:
- **Bežní používatelia**: Žiadne úpravy, štandardný priebeh
- **Uprednostnení používatelia**: Prepíše "žiadna dostupnosť" na "dostupné"
- **Podmienená logika**: Prepis len keď je potrebný

#### **4. Rovnaký priebeh, iné výsledky**

Sila middleware:
- ✅ Žiadne zmeny v štruktúre priebehu
- ✅ Žiadne zmeny vo funkcii nástroja
- ✅ Žiadne zmeny v logike podmieneného smerovania
- ✅ Iba middleware → Iné správanie!

### 🚀 Skutočné použitia:

1. **VIP/Premium funkcie**
   - Prepísanie limitov pre prémiových používateľov
   - Poskytnutie prioritného prístupu k zdrojom
   - Dynamické odomknutie prémiových funkcií

2. **A/B testovanie**
   - Smerovanie používateľov na rôzne implementácie
   - Testovanie nových funkcií so špecifickými používateľmi
   - Postupné zavádzanie funkcií

3. **Bezpečnosť a súlad s pravidlami**
   - Audit volaní funkcií
   - Blokovanie citlivých operácií
   - Vynucovanie biznis pravidiel

4. **Optimalizácia výkonu**
   - Ukladanie výsledkov do cache pre vybraných používateľov
   - Preskakovanie nákladných operácií keď je to možné
   - Dynamické prideľovanie zdrojov

5. **Spracovanie chýb a opakovanie**
   - Zachytávanie a elegantné spracovanie chýb
   - Implementácia logiky opakovania
   - Náhradné riešenia pomocou alternatívnych implementácií

6. **Logovanie a sledovanie**
   - Sledovanie doby vykonávania funkcie
   - Logovanie parametrov a výsledkov
   - Monitorovanie vzorov využívania

### 🔑 Hlavné rozdiely oproti dekorátorom:

| Vlastnosť | Dekorátor | Middleware |
|-----------|-----------|------------|
| **Rozsah** | Jedna funkcia | Všetky funkcie agenta |
| **Flexibilita** | Fixná pri definícii | Dynamická počas behu |
| **Kontext** | Obmedzený | Plný kontext agenta |
| **Kombinácia** | Viac dekorátorov | Middleware pipeline |
| **Agent-aware** | Nie | Áno (prístup k stavu agenta) |

### 📚 Kedy použiť middleware:

✅ **Použite middleware keď:**
- Potrebujete meniť správanie na základe stavu používateľa/sedenia
- Chcete aplikovať logiku na viacero funkcií
- Potrebujete prístup ku kontextu na úrovni agenta
- Implementujete prierezové problémy (logovanie, autentifikácia, atď.)

❌ **Nepoužívajte middleware keď:**
- Jednoduchá validácia vstupov (použite Pydantic)
- Logika špecifická pre funkciu (zostaňte vo funkcii)
- Jednorazové úpravy (zmeňte priamo funkciu)

### 🎓 Pokročilé vzory:

```python
# Viaceré middleware (poradie vykonávania je dôležité!)
middleware=[
    logging_middleware,      # Najprv záznamy
    auth_middleware,         # Potom kontrola autentifikácie
    cache_middleware,        # Potom kontrola cache
    rate_limit_middleware,   # Potom obmedzenie rýchlosti
    priority_check_middleware  # Nakoniec kontrola priority
]

# Podmienené vykonávanie middleware
async def conditional_middleware(context, next):
    if should_execute(context):
        await next(context)
        # Upraviť výsledok
    else:
        # Úplné preskočenie vykonávania
        context.result = cached_value
```

### 🔗 Súvisiace koncepty:

- **Agent Middleware**: Zachytáva volania agent.run()
- **Function Middleware**: Zachytáva volania funkcií nástrojov (čo sme používali!)
- **Middleware Pipeline**: Reťaz middleware spúšťaných postupne
- **Context Propagation**: Prenos stavu cez middleware reťazec


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Vyhlásenie o zodpovednosti**:
Tento dokument bol preložený pomocou AI prekladateľskej služby [Co-op Translator](https://github.com/Azure/co-op-translator). Hoci sa snažíme o presnosť, vezmite prosím na vedomie, že automatické preklady môžu obsahovať chyby alebo nepresnosti. Pôvodný dokument v jeho natívnom jazyku by mal byť považovaný za autoritatívny zdroj. Pre kritické informácie sa odporúča profesionálny ľudský preklad. Nie sme zodpovední za žiadne nedorozumenia alebo nesprávne interpretácie vyplývajúce z použitia tohto prekladu.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
